In [2]:
from abc import ABC, abstractmethod
from decimal import Decimal


from mainnet_launch.constants import ChainData, ETH_CHAIN
from multicall import Call
from mainnet_launch.data_fetching.get_state_by_block import get_state_by_one_block, build_blocks_to_use
import pandas as pd
import plotly.express as px
import plotly.io as pio

SECONDS_PER_YEAR = 365 * 24 * 60 * 60


def get_seconds_between_blocks(
    chain: ChainData,
    start_block: int,
    end_block: int,
) -> int:
    start_timestamp = chain.client.eth.get_block(start_block)["timestamp"]
    end_timestamp = chain.client.eth.get_block(end_block)["timestamp"]
    return end_timestamp - start_timestamp


class BacktestBaseAndFeeAPRCalculator(ABC):

    def __init__(self, chain: ChainData):
        self.chain = chain

    @abstractmethod
    def _get_rate(self, block: int) -> int:
        """
        Returns exchange rate, virtual price, convertToAssets(1e18) or similar depending on the destination
        """
        pass

    def compute_base_and_fee_apr(self, start_block: int, end_block: int) -> float:
        """
        Returns the base APR in portion form. eg .01 for 1% APR between start and end block
        """
        start_rate = self._get_rate(start_block)
        end_rate = self._get_rate(end_block)

        rate_delta = end_rate - start_rate
        seconds_between_blocks = get_seconds_between_blocks(self.chain, start_block, end_block)
        portion_of_year = seconds_between_blocks / SECONDS_PER_YEAR
        base_apr = (rate_delta / start_rate) / portion_of_year
        return base_apr


class BalancerBacktestBaseAndFeeAPRCalculator(BacktestBaseAndFeeAPRCalculator):
    def __init__(self, chain: ChainData, pool_address: str):
        super().__init__(chain)
        self.pool_address = pool_address

    def _get_rate(self, block: int) -> int:
        """
        Returns "rate" invarient / bpt total supply
        """

        def identify_function(success, x):
            if success:
                return x

        call = Call(
            self.pool_address,
            ["getRate()(uint256)"],
            [["getRate", identify_function]],
        )
        state = get_state_by_one_block([call], block, self.chain)
        return state["getRate"]

/Users/pb/Library/Caches/pypoetry/virtualenvs/mainnet-launch-FtycU18g-py3.10/lib/python3.10/site-packages/web3/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
pool_address = "0x6b31a94029fd7840d780191B6D63Fa0D269bd883"
surge_fluid_wsteth_weth_balancer_pool = BalancerBacktestBaseAndFeeAPRCalculator(
    chain=ETH_CHAIN, pool_address=pool_address
)

# a block at the end of each day since the pool was deployed
blocks = build_blocks_to_use(start_block=22124378, chain=ETH_CHAIN)

# Get the rate at each block to track historical values
historical_rates = []
for block in blocks:
    rate = surge_fluid_wsteth_weth_balancer_pool._get_rate(block)
    historical_rates.append({"block": block, "rate": rate})


records = []

for i, start_block in enumerate(blocks[:-1]):
    end_block = blocks[i + 1]
    fee_and_base_apr = surge_fluid_wsteth_weth_balancer_pool.compute_base_and_fee_apr(start_block, end_block)
    start_datetime = pd.to_datetime(ETH_CHAIN.client.eth.get_block(start_block)["timestamp"], unit="s", utc=True)
    records.append(
        {
            "start_datetime": start_datetime,
            "start_block": start_block,
            "end_block": end_block,
            "fee_and_base_apr": fee_and_base_apr,
        }
    )

In [4]:
import plotly.express as px
import plotly.io as pio

import pandas as pd

pio.templates.default = "plotly_white"

df = pd.DataFrame.from_records(records).sort_values("end_block")
px.scatter(df, x="start_datetime", y="fee_and_base_apr", title="Surge Fluid wstETH/WETH Balancer Pool Fee + Base APR")